In [1]:
import os

In [2]:
import numpy as np
import sys
import csv
from IPython.display import clear_output
from stable_baselines3 import DQN
import requests
import datetime
import torch

In [3]:
clear_output(wait=True)

In [4]:
url_api = 'https://api.boptest.net'
url = "http://localhost:80"
log_path = os.path.join("Logs")

In [5]:
sys.path.insert(0,'../boptestGym')
from boptestGymEnv import BoptestGymEnv
from boptestGymEnv import NormalizedObservationWrapper
from boptestGymEnv import DiscretizedActionWrapper

In [6]:
def denormalize(norm_vals, lo, hi): 
    return 0.5 * (norm_vals + 1.0) * (hi - lo) + lo

kelvin = lambda c: c + 273.15
celc = lambda k: k -273.15

In [7]:
bounds = {
    "T_in": (273,324),
    "T_out": (257,305),
    "Psol": (0, 862),
}

step_period_h = 0.5
floor_area = 16 * 12
low_setpoint_k  = kelvin(21.0)
high_setpoint_k = kelvin(24.0)

In [8]:
class BoptestGymEnvCustomReward(BoptestGymEnv):
    def reset(self, *args, **kwargs):
        obs, info = super().reset(**kwargs)
        self.last_obs = obs
        self.last_action=0
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        self.last_obs = obs
        self.last_action = action
        return obs, reward, terminated, truncated, info
    
    def get_reward(self):
        obs = self.last_obs
        action = self.last_action
        T_in_K_val = obs[0]
        discomfort = (max(0.0, low_setpoint_k  - T_in_K_val) +
                  max(0.0, T_in_K_val - high_setpoint_k)) * step_period_h
        energy = float(action) * 15.0 * step_period_h  / floor_area
        reward = -(discomfort + energy)

        reward += self.objective_integrand
        self.objective_integrand = reward
        return reward

In [9]:
feb15 = 47 * 24*3600                    # Jan 1 → Feb 15 low data regime
june30 = 181 * 24 *3600                 # high data regime
episode_length = 7 * 24*3600            # 1 week episodes

In [10]:
def create_env(max_start):
    env = BoptestGymEnvCustomReward(url  = url,
                    testcase              = 'bestest_hydronic_heat_pump',
                    actions               = ['oveHeaPumY_u'],
                    observations          = {'reaTZon_y':(273.,324.), #0 / 50
                                             'weaSta_reaWeaTDryBul_y':(257.,305.), # -15 / 28.8
                                             'weaSta_reaWeaHDirNor_y':(0.,862.)
                                            },
                    random_start_time     = True,
                    max_episode_length    = 7 * 24*3600,
                    excluding_periods     = [(max_start, 365*24*3600)],
                    warmup_period         = 24*3600,
                    predictive_period     = 0,
                    regressive_period     = 4*1800,
                    step_period           = 1800
                               )
    env = NormalizedObservationWrapper(env)
    env = DiscretizedActionWrapper(env,n_bins_act=1)
    print("env created")
    return env

In [11]:
for data_regime in [6, 27]:
    for training_episodes in [25, 50]:
        print(f"training episode: {training_episodes}")
        if data_regime == 6:
            max_start = feb15 - episode_length      # last valid start time
            print("6 weeks")
        elif data_regime == 27:
            max_start = june30 - episode_length      # last valid start time
            print("27 weeks")
        env = create_env(max_start)    
        policy_kwargs = dict(
            net_arch=[64, 8],  
            activation_fn=torch.nn.ReLU
        )

        model = DQN('MlpPolicy',
            env,
            verbose=1,
            gamma=0.99,
            learning_rate=0.01,
            batch_size=64,
            buffer_size=20000,
            learning_starts=0,
            train_freq=1,
            target_update_interval=1000,
            tau=1.0,
            gradient_steps=1,
            exploration_fraction=0.1,          #ε-greedy?
            exploration_initial_eps=1.0,
            exploration_final_eps=0.05,
            policy_kwargs=policy_kwargs,
            tensorboard_log= log_path)
        
        total_timesteps = 336 * training_episodes 
        model.learn(total_timesteps= total_timesteps, tb_log_name = f"DQN_switch_1_{data_regime}_{training_episodes}") 
        model.save(f"Saved Models/DQN_switch_1_{data_regime}_{training_episodes}")
        print(f"model saved at: Saved Models/DQN_switch_1_{data_regime}_{training_episodes}")
        buffer_path = os.path.join( "Memory Buffers", f"DQN_switch_1_{data_regime}_{training_episodes}")
        model.save_replay_buffer(buffer_path)
        print("memory buffer saved")
        env.stop()
        del model

training episode: 25
6 weeks


C:\Users\irmak\anaconda3\Lib\site-packages\gymnasium\spaces\box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


env created
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to Logs\DQN_switch_1_6_25_2


C:\Users\irmak\AppData\Local\Temp\ipykernel_17372\399620426.py:20: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  energy = float(action) * 15.0 * step_period_h  / floor_area


-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 336       |
|    ep_rew_mean      | -8.93e+04 |
|    exploration_rate | 0.05      |
| time/               |           |
|    episodes         | 4         |
|    fps              | 16        |
|    time_elapsed     | 83        |
|    total_timesteps  | 1344      |
| train/              |           |
|    learning_rate    | 0.01      |
|    loss             | 91.5      |
|    n_updates        | 1343      |
-----------------------------------
-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 336       |
|    ep_rew_mean      | -1.11e+05 |
|    exploration_rate | 0.05      |
| time/               |           |
|    episodes         | 8         |
|    fps              | 16        |
|    time_elapsed     | 163       |
|    total_timesteps  | 2688      |
| train/              |           |
|    learning_rate    | 0.01      |
|    loss             | 98.8

C:\Users\irmak\anaconda3\Lib\site-packages\gymnasium\spaces\box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


env created
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to Logs\DQN_switch_1_6_50_2
-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 336       |
|    ep_rew_mean      | -1.11e+04 |
|    exploration_rate | 0.24      |
| time/               |           |
|    episodes         | 4         |
|    fps              | 16        |
|    time_elapsed     | 79        |
|    total_timesteps  | 1344      |
| train/              |           |
|    learning_rate    | 0.01      |
|    loss             | 14.3      |
|    n_updates        | 1343      |
-----------------------------------
-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 336       |
|    ep_rew_mean      | -8.68e+03 |
|    exploration_rate | 0.05      |
| time/               |           |
|    episodes         | 8         |
|    fps              | 17        |
|    time_elapsed     | 156  